In [50]:
!pip install google-adk litellm httpx

In [51]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"]=userdata.get("GOOGLE_API_KEY")

In [52]:
import httpx

In [53]:
import os
from google.colab import userdata

os.environ["AZURE_OPENAI_API_KEY"] = userdata.get('AZURE_OPENAI_API_KEY')
os.environ["AZURE_OPENAI_ENDPOINT"] =userdata.get("AZURE_OPENAI_ENDPOINT")
os.environ["OPENAI_API_VERSION"] = "2024-12-01-preview"

In [54]:
from google.adk.models.lite_llm import LiteLlm
aoai_model = LiteLlm(
    model="azure/gpt-5.4-mini",
    api_base=os.environ["AZURE_OPENAI_ENDPOINT"]
)

In [55]:
def resolve_dns(hostname: str, record_type: str = "A") -> dict:
    """
    Perform a real DNS lookup using Google Public DNS-over-HTTPS. Debug infra, check propagation.

    Args:
        hostname: Domain to resolve, e.g. "api.github.com", "npmjs.com"
        record_type: DNS record type — "A", "AAAA", "CNAME", "MX", "TXT", "NS" (default: "A")

    Returns:
        Resolved DNS records with TTL and values.
    """
    try:
        with httpx.Client(timeout=10) as client:
            r = client.get(
                "https://dns.google/resolve",
                params={"name": hostname, "type": record_type},
                headers={"Accept": "application/dns-json"}
            )
            r.raise_for_status()
            data = r.json()

        status_map = {0: "NOERROR", 1: "FORMERR", 2: "SERVFAIL", 3: "NXDOMAIN"}
        status = status_map.get(data.get("Status", -1), f"CODE_{data.get('Status')}")

        answers = [
            {"name": rec.get("name"), "type": record_type, "ttl_seconds": rec.get("TTL"), "value": rec.get("data")}
            for rec in data.get("Answer", [])
        ]

        return {
            "query": hostname,
            "record_type": record_type,
            "status": status,
            "answers": answers,
            "answer_count": len(answers),
        }
    except Exception as e:
        return {"error": str(e)}



In [56]:

def get_pypi_package_info(package_name: str) -> dict:
    """
    Fetch Python package details from PyPI: version, author, license, deps, install command.

    Args:
        package_name: PyPI package name, e.g. "google-adk", "fastapi", "langchain", "httpx"

    Returns:
        Latest version, summary, author, license, Python requirement, and top dependencies.
    """
    try:
        with httpx.Client(timeout=10) as client:
            r = client.get(f"https://pypi.org/pypi/{package_name}/json")
            r.raise_for_status()
            info = r.json()["info"]

        return {
            "name": info.get("name"),
            "latest_version": info.get("version"),
            "summary": info.get("summary"),
            "author": info.get("author") or info.get("author_email"),
            "license": info.get("license", "Unknown"),
            "homepage": info.get("home_page") or "",
            "requires_python": info.get("requires_python", "Not specified"),
            "requires_dist": (info.get("requires_dist") or [])[:10],
            "classifiers": [
                c for c in (info.get("classifiers") or [])
                if "Programming Language" in c or "Framework" in c
            ][:5],
            "install_command": f"pip install {info.get('name')}",
            "pypi_url": f"https://pypi.org/project/{info.get('name')}/",
        }
    except Exception as e:
        return {"error": str(e)}



In [57]:
from google.adk.agents.llm_agent import Agent

root_agent = Agent(
    model=aoai_model,
    name="tech_assistant_agent",
    description="A developer-focused assistant with real-time access to PyPI, DNS.",
    instruction="""
You are a skilled tech assistant with access to real-time developer tools.

## Your Tools

1. **resolve_dns** — Resolve DNS records (A, CNAME, MX, TXT, NS) for any domain.
2. **get_pypi_package_info** — Get Python package info from PyPI.

## Behavior Rules

- Always call the relevant tool before answering — never guess at versions or DNS records.
- For comparisons (e.g. "fastapi vs flask"), call the tool for each package.
- Format large numbers: 45200 stars → "45.2k ⭐", 3100000 downloads → "3.1M/week".
- If a tool errors, explain it and suggest an alternative.
- Be concise and technically precise.
""",
    tools=[
        resolve_dns,
        get_pypi_package_info,
    ],
)

In [58]:
from google.adk.runners import InMemoryRunner
runner=InMemoryRunner(agent=root_agent)
response=await runner.run_debug("What are the DNS records for google.com??")


 ### Created new session: debug_session_id

User > What are the DNS records for google.com??

Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers

tech_assistant_agent > Here are the current DNS records for **google.com**:

**A**
- 142.250.98.102
- 142.250.98.138
- 142.250.98.113
- 142.250.98.101
- 142.250.98.100
- 142.250.98.139

**AAAA**
- 2607:f8b0:400c:c01::8b
- 2607:f8b0:400c:c01::65
- 2607:f8b0:400c:c01::8a
- 2607:f8b0:400c:c01::66

**CNAME**
- None

**MX**
- 10 smtp.google.com.

**TXT**
- google-site-verification=wD8N7i1JTNTkezJ49swvWW48f8_9xveREV4oB-0Hf5o
- cisco-ci-domain-verification=47c38bc8c4b74b7233e9053220c1bbe76bcc1cd33c7acf7acd36cd6a5332004b
- MS=E4A68B9AB2BB9670BCE15412F62916164C0B20BB
- onetrust-domain-verification=0d477fe608074e6f9c12bca7826035cc
- facebook-domain-verification=22rm551cu4k0ab0bxsw536tlds4h95
- apple-domain-verification=30afIBcvSuDV2PLX
- 

In [59]:
from google.adk.runners import InMemoryRunner
runner=InMemoryRunner(agent=root_agent)
response=await runner.run_debug("Tell me about the requests package on PyPI.")


 ### Created new session: debug_session_id

User > Tell me about the requests package on PyPI.

Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers

tech_assistant_agent > `requests` is a popular Python HTTP client for making web requests.

- Latest version: **2.33.1**
- Summary: **Python HTTP for Humans.**
- Author: **Kenneth Reitz**
- License: **Apache-2.0**
- Requires Python: **>=3.10**

Dependencies:
- `charset_normalizer<4,>=2`
- `idna<4,>=2.5`
- `urllib3<3,>=1.26`
- `certifi>=2023.5.7`

Extras:
- `socks` support via `PySocks`
- optional `use-chardet-on-py3`

Install:
```bash
pip install requests
```

PyPI: https://pypi.org/project/requests/

If you want, I can also compare `requests` with `httpx` or show a quick usage example.


Google Search: "What is the latest news on AI in healthcare?"

Resolve DNS: "What are the DNS records for google.com?"

PyPI Package Info: "Tell me about the requests package on PyPI."